## What to Vary

In [1]:
# language="english"
# language="multilingual"
# DeepPavlov/rubert-base-cased-sentence


# raw text or vw text


# default topics (whatever)
# specific number of topics


# KeyBERTInspired
# openchat

In [2]:
from topicnet.cooking_machine import Dataset

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

from umap import UMAP
from hdbscan import HDBSCAN

from hdbscan.flat import HDBSCAN_flat

from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora

from gensim.models.coherencemodel import CoherenceModel

import pandas as pd

In [3]:
import nltk
from nltk.corpus import stopwords
 
nltk.download('stopwords')

print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alekseev_v/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/ruwiki_good.txt',
)

dataset.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [7]:
MAIN_MODALITY = '@lemmatized'

In [8]:
dataset._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [9]:
dataset._data['raw_text']

id
Санкт-Петербург                         
Дворцовая_площадь                       
Греко-персидские_войны                  
Тихий_океан                             
Атлантический_океан                     
                                      ..
Бассет,_Филипп                          
Битва_при_Линкольне_(1217)              
Лю_Жэньхан                              
Реформа_эталонных_процентных_ставок     
Ментеше-бей                             
Name: raw_text, Length: 8603, dtype: object

In [10]:
dataset._data.shape

(8603, 3)

In [11]:
dataset._data.dropna(axis=0, inplace=True)

In [12]:
dataset._data.shape

(8603, 3)

In [13]:
dataset._data['vw_text']

id
Санкт-Петербург                        Санкт-Петербург |@lemmatized год:301 петроград...
Дворцовая_площадь                      Дворцовая_площадь |@lemmatized дворцовый:43 пл...
Греко-персидские_войны                 Греко-персидские_войны |@lemmatized грёкий:23 ...
Тихий_океан                            Тихий_океан |@lemmatized тихий:92 океан:174 ус...
Атлантический_океан                    Атлантический_океан |@lemmatized атлантический...
                                                             ...                        
Бассет,_Филипп                         Бассет,_Филипп |@lemmatized бассет:44 <person>...
Битва_при_Линкольне_(1217)             Битва_при_Линкольне_(1217) |@lemmatized битва:...
Лю_Жэньхан                             Лю_Жэньхан |@lemmatized лю:29 жэньхан:25 китай...
Реформа_эталонных_процентных_ставок    Реформа_эталонных_процентных_ставок |@lemmatiz...
Ментеше-бей                            Ментеше-бей |@lemmatized ?:1 умереть:2 основат...
Name: vw_text, Len

In [14]:
dataset._data['vw_text'][0]

'Санкт-Петербург |@lemmatized год:301 петроград:7 ленинград:18 численность:14 население:33 город:212 россия:32 .:677 федеральный:15 значение:8 административный:5 центр:29 округа:3 ленинградский:16 область:6 основать:5 царь:3 <person>:195 являться:34 столица:19 российский:35 государство:9 назвать:4 честь:6 святой:11 небесный:2 покровитель:2 основатель:3 время:17 стать:25 большой:23 ассоциироваться:1 имя:34 исторически:1 культурно:1 связать:2 рождение:1 империя:7 вхождение:1 современный:7 история:11 роль:5 европейский:3 великий:5 держава:1 расположить:11 страна:16 побережье:4 финский:16 залив:15 устье:4 река:21 нева:31 находиться:17 конституционный:1 суд:5 федерация:14 геральдический:2 совет:8 президент:1 орган:6 власть:11 межпарламентский:3 ассамблея:3 снг:2 разместить:1 главный:9 командование:2 флот:1 штаб:2 западный:8 военный:8 вооружённый:3 сила:6 быть:74 революция:6 февральский:2 октябрьский:6 ход:4 отечественный:4 война:10 блокада:7 результат:12 миллион:37 человек:39 погибнуть:4 об

In [15]:
import re

def get_modality_tokens(text, modality):
    m = re.search(fr'\|{modality} ([^@]+) \|@', text)
    vw_text = m.groups()[0]

    text = ''

    for token_freq in vw_text.split():
        token, freq = token_freq.split(':')

        if freq == '':
            freq = 1
        else:
            freq = int(freq)

        text += (token + ' ') * freq

    return text.strip()

In [16]:
get_modality_tokens(dataset._data['vw_text'][0], MAIN_MODALITY)

'год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год

In [17]:
get_modality_tokens(dataset._data['vw_text'][0], MAIN_MODALITY)

'год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год

In [18]:
docs = [get_modality_tokens(d, MAIN_MODALITY) for d in dataset._data['vw_text']]

In [19]:
len(docs)

8603

In [20]:
docs[:3]

['год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год год го

In [21]:
NUM_TOP_WORDS = 20

In [22]:
import torch
import transformers
import os

import json
import numpy as np

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [23]:
def get_phi(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    # ptw = np.array(mtw[1:, :])
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T
    vocabulary = topic_model.vectorizer_model.get_feature_names_out()

    assert pwt.shape[0] == len(vocabulary)

    # topic_names = [f'topic_{i}' for i in range(pwt.shape[1])]
    # topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(pwt.shape[1] - 1)]

    phi = pd.DataFrame(
        index=vocabulary,
        columns=topic_names,
        data=pwt,
    )

    return phi


def get_top_words(topic_model):
    mtw = topic_model.c_tf_idf_.todense()
    ptw = np.array(mtw[0:, :])
    pwt = ptw.T

    # topic_names = ['background_1'] + [f'topic_{i}' for i in range(NUM_TOPICS)]

    # TODO: lost topic in top words, "background -> False", because does not have -1 topic for GoodRuWiki
    topic_names = ['background_1'] + [f'topic_{i}' for i in range(pwt.shape[1] - 1)]
    topic_top_words = {
        n: topic_model.get_topic(t)
        for t, n in zip([-1] + list(range(NUM_TOPICS)), topic_names)
    }

    return topic_top_words


def get_dataset(topic_model, dataset, docs):
    cleaned_docs = topic_model._preprocess_text(docs)
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()
    doc_tokens = [tokenizer(doc) for doc in cleaned_docs]
    doc_texts = [
        d + f' |{MAIN_MODALITY} ' + ' '.join(t)
        for d, t in zip(dataset._data.index, doc_tokens)
    ]
    data = [[d, t] for d, t in zip(dataset._data.index, doc_texts)]

    new_dataset = pd.DataFrame(
        columns=['id', 'vw_text'],
        data=data,
    )

    return new_dataset

In [24]:
NUM_TOPICS = 20
NUM_TOP_WORDS = 20
NUM_TRAINS = 20
STOP_WORDS = stopwords.words('russian')
LANGUAGE = -1  # 'multilingual'

In [25]:
! ls ../results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [26]:
SAVE_FOLDER = os.path.join('/data_mil/shared/CompressaAI/BERTopic', 'results', 'ruwikigood')

In [27]:
! mkdir -p $SAVE_FOLDER

In [28]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood'

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

embedding_model = TfidfVectorizer(min_df=5, max_df=int(len(docs) * 0.5), stop_words=STOP_WORDS)
embeddings = embedding_model.fit_transform(docs)
# topic_model = BERTopic().fit(docs, embeddings)

In [30]:
len(docs)

8603

In [31]:
embeddings.shape

(8603, 61682)

In [ ]:
embeddings = TfidfVectorizer(stop_words=STOP_WORDS).fit_transform(docs)

In [50]:
embeddings.shape

(8603, 264925)

In [38]:
for seed in [100]:  # range(NUM_TRAINS):
    print(seed)

    seed_save_folder = os.path.join(SAVE_FOLDER, str(seed))

    if os.path.isdir(seed_save_folder):
        contents = os.listdir(seed_save_folder)

        assert len(contents) == 3

        continue

    # TODO: os.makedirs(seed_save_folder)

    keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
    mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)
    
    representation_model = {
        "KeyBERT": keybert,
        "MMR": mmr,
    }

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,

        calculate_probabilities=True,
        verbose=True,

        # embedding_model=embedding_model,          # Step 1 - Extract embeddings
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
                                                  # Step 5 - Extract topic words
        # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )
        
    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
    orig_num_topics = len(set(topic_model.topics_))
    doc_embeddings = topic_model.umap_model.embedding_

    hdbscan_model = HDBSCAN_flat(doc_embeddings, n_clusters=NUM_TOPICS)
    
    topic_model = BERTopic(
        language=LANGUAGE,
        top_n_words=NUM_TOP_WORDS,
        calculate_probabilities=True,
        verbose=True,
    
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
    )

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
    
    new_num_topics = len(set(topic_model.topics_))
    
    # assert new_num_topics < orig_num_topics
    if new_num_topics >= orig_num_topics:
        print(f'No less topics: {new_num_topics} >= {orig_num_topics}.')

    # assert new_num_topics == NUM_TOPICS + 
    if new_num_topics != NUM_TOPICS + 1:
        print(f'WTF: failed to produce exact number of topics: {new_num_topics} != {NUM_TOPICS + 1}.')

        # whatever
        # assert abs(new_num_topics - (NUM_TOPICS + 1)) <= 2

    assert topic_model.c_tf_idf_.shape[0] == new_num_topics
    
    phi = get_phi(topic_model)
    top_words = get_top_words(topic_model)
    new_dataset = get_dataset(topic_model, dataset, docs)

    # TODO assert Falsse
    
    phi.to_csv(f'{seed_save_folder}/phi.csv')
    
    with open(f'{seed_save_folder}/top_words.json', 'w') as f:
        f.write(
            json.dumps(
                top_words, indent=4, ensure_ascii=False
            )
        )
    
    new_dataset.to_csv(f'{seed_save_folder}/dataset.csv')

    del topic_model, phi, new_dataset

2024-03-30 17:26:23,604 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


100


2024-03-30 17:27:59,448 - BERTopic - Dimensionality - Completed ✓
2024-03-30 17:27:59,450 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 17:28:07,560 - BERTopic - Cluster - Completed ✓
2024-03-30 17:28:07,566 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 17:28:30,947 - BERTopic - Representation - Completed ✓
2024-03-30 17:28:43,893 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 17:30:19,544 - BERTopic - Dimensionality - Completed ✓
2024-03-30 17:30:19,546 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 17:30:42,024 - BERTopic - Cluster - Completed ✓
2024-03-30 17:30:42,028 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 17:31:04,683 - BERTopic - Representation - Completed ✓


WTF: failed to produce exact number of topics: 17 != 21.


NameError: name 'Fale' is not defined

In [46]:
set(topics)

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16}

In [48]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,5450,0_person_год_время_стать,"[person, год, время, стать, город, человек, ча...",[<person> <person> <person> <person> <person> ...
1,1,706,1_вид_клетка_мочь_птица,"[вид, клетка, мочь, птица, акула, являться, pe...",[насекомое насекомое насекомое насекомое насек...
2,2,525,2_person_консул_год_рим,"[person, консул, год, рим, цезарь, римский, кв...",[<person> <person> <person> <person> <person> ...
3,3,411,3_матч_команда_person_сезон,"[матч, команда, person, сезон, клуб, год, чемп...",[<person> <person> <person> <person> <person> ...
4,4,392,4_альбом_группа_песня_person,"[альбом, группа, песня, person, песнь, год, му...",[beatles beatles beatles beatles beatles beatl...
5,5,308,5_картина_художник_person_год,"[картина, художник, person, год, полотно, порт...",[<person> <person> <person> <person> <person> ...
6,6,286,6_корабль_крейсер_мм_флот,"[корабль, крейсер, мм, флот, орудие, год, pers...",[линейный линейный линейный линейный линейный ...
7,7,101,7_монета_марка_денежный_год,"[монета, марка, денежный, год, монетный, номин...",[торговый торговый торговый торговый торговый ...
8,8,92,8_станция_паровоз_поезд_вагон,"[станция, паровоз, поезд, вагон, линия, год, д...",[александровский александровский александровск...
9,9,73,9_sonic_игра_hedgehog_соника,"[sonic, игра, hedgehog, соника, уровень, персо...",[sonic sonic sonic sonic sonic sonic sonic son...


In [47]:
probs

array([[4.33703841e-004, 4.12765117e-004, 3.04115154e-004, ...,
        3.11624880e-004, 2.73035668e-004, 4.79422263e-004],
       [8.52105858e-004, 8.04528964e-004, 6.21072604e-004, ...,
        6.48838103e-004, 5.49378073e-004, 9.41348441e-004],
       [2.51743835e-004, 2.27136198e-004, 2.32856088e-004, ...,
        2.27431234e-004, 1.90792924e-004, 2.74033365e-004],
       ...,
       [5.36562846e-004, 4.98457130e-004, 4.98985329e-004, ...,
        4.36115598e-004, 4.19582198e-004, 5.97808717e-004],
       [4.16437041e-004, 3.83560639e-004, 3.05152769e-004, ...,
        2.74464943e-004, 2.63524561e-004, 4.69710815e-004],
       [7.97832361e-308, 7.51012599e-308, 7.57606958e-308, ...,
        7.48443426e-308, 7.04970891e-308, 8.58309278e-308]])

In [39]:
phi = get_phi(topic_model)
top_words = get_top_words(topic_model)
new_dataset = get_dataset(topic_model, dataset, docs)

In [40]:
phi.head()

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15
aa,0.000021,0.000066,0.0,0.00000,0.000016,0.0,0.000019,0.0,0.0,0.0,0.0,0.0,0.007627,0.0,0.0,0.0,0.0
aaa,0.000011,0.000011,0.0,0.00003,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaaa,0.000003,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaaabbbbbabbbaabbababbaaababaab,0.000003,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaab,0.000004,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [41]:
phi.columns

Index(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4',
       'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10',
       'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'],
      dtype='object')

In [42]:
top_words

{'background_1': False,
 'topic_0': [('person', 0.042461024327904114),
  ('год', 0.028026665769274293),
  ('время', 0.012182631252993337),
  ('стать', 0.010203728015430421),
  ('город', 0.008503057248634838),
  ('человек', 0.008454315951511773),
  ('часть', 0.007892679519769156),
  ('имя', 0.007234802031268627),
  ('мочь', 0.007187300486495405),
  ('являться', 0.007061139441013748),
  ('работа', 0.007048734715160412),
  ('век', 0.006955706259198057),
  ('получить', 0.006824536172941751),
  ('иметь', 0.006543076987295329),
  ('игра', 0.006439551874001519),
  ('большой', 0.006359507365025279),
  ('армия', 0.006238640861521342),
  ('война', 0.006006533723150852),
  ('фильм', 0.005999963673066351),
  ('новый', 0.005694016832708471)],
 'topic_1': [('вид', 0.03034024096957983),
  ('клетка', 0.025942453308010077),
  ('мочь', 0.02211897797194643),
  ('птица', 0.020593829955137824),
  ('акула', 0.016642959693217083),
  ('являться', 0.01444510116705008),
  ('person', 0.014318273721880692),
  ('г

In [49]:
# Wow! Where is -1 topic?..

In [44]:
topic_model.get_topic(-1)

False

In [73]:
! ls $SAVE_FOLDER

0  1  10  11  12  13  14  15  16  17  18  19  2  3  4  5  6  7	8  9


In [45]:
new_num_topics

52

In [74]:
SAVE_FOLDER

'/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood'

In [54]:
! ls $SAVE_FOLDER

0  1  10  11  2  3  4  5  6  7	8  9


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
! ls /data_mil/shared/CompressaAI/BERTopic/results50/rtlwikiperson/

0  1  10  11  2  3  4  5  6  7	8  9


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [48]:
! ls /data_mil/shared/CompressaAI/BERTopic/results50/rtlwikiperson/11

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [33]:
keybert = KeyBERTInspired(top_n_words=NUM_TOP_WORDS)
mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=NUM_TOP_WORDS)

representation_model = {
    "KeyBERT": keybert,
    "MMR": mmr,
}

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
vectorizer_model = CountVectorizer(stop_words=STOP_WORDS)

topic_model = BERTopic(
    language=LANGUAGE,
    top_n_words=NUM_TOP_WORDS,

    calculate_probabilities=True,
    verbose=True,

    # embedding_model=embedding_model,          # Step 1 - Extract embeddings
    umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    # hdbscan_model=hdbscan_model,            # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
                                              # Step 5 - Extract topic words
    # representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)
    
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

2024-03-30 17:21:25,071 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-30 17:23:18,990 - BERTopic - Dimensionality - Completed ✓
2024-03-30 17:23:18,992 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-30 17:23:28,561 - BERTopic - Cluster - Completed ✓
2024-03-30 17:23:28,572 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-30 17:23:51,066 - BERTopic - Representation - Completed ✓


In [35]:
len(set(topics))

171

In [37]:
topic_model.get_topic(1)

[('картина', 0.03225299001920325),
 ('художник', 0.029269929925598223),
 ('полотно', 0.014258702951247364),
 ('портрет', 0.012852161036762714),
 ('выставка', 0.011311901541073366),
 ('искусство', 0.00945169711855185),
 ('живопись', 0.009012511979103249),
 ('искусствовед', 0.00851818232744616),
 ('изобразить', 0.00809599584320772),
 ('работа', 0.007972387468817748),
 ('музей', 0.007765781939114305),
 ('фигура', 0.0076978975667414355),
 ('галерея', 0.0074293124118641),
 ('художественный', 0.007427091181182913),
 ('шахматы', 0.007046947817575379),
 ('person', 0.006934069498057664),
 ('шахматный', 0.006474895599185693),
 ('холст', 0.006310108934025328),
 ('изображение', 0.005870506547391321),
 ('год', 0.005650590902580173)]